[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GNash19/paminaw/blob/v1.0.0/notebooks/solutions/01_load_and_see.ipynb)

# Paminaw! · Lab 1 · Load & see (solutions)

A standalone copy of one lab from the PyCon Davao 2026 workshop *Speech Processing Using
PyTorch* (Nash Golosino and Aurora Manseras), for study after the event. Load recordings as tensors, resample, and compute mel spectrograms.

Run **cell 0** first, then go from top to bottom with **Shift+Enter**. Scores are on the
**workshop subset (100 clips per language)**, not the paper's full-split numbers.

## 0 · Setup: run me first

This cell installs the workshop package `paminaw`, prints the versions of the libraries we
use, checks for a GPU and downloads the 200 audio clips. Wait for the **✅ READY** line
(about a minute). If something fails, the message tells you how to fix it.

In [ ]:
# Cell 0 · setup
import os
import subprocess
import sys

PAMINAW_REF = "v1.0.0"  # the workshop release on GitHub
SMOKE_TEST = os.environ.get("PAMINAW_SMOKE") == "1"  # facilitators: 2 clips, whisper-tiny

if "google.colab" in sys.modules:
    print(f"Installing paminaw {PAMINAW_REF} ...")
    url = f"paminaw @ git+https://github.com/GNash19/paminaw@{PAMINAW_REF}"
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", url],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise SystemExit("❌ Could not install paminaw. Fix: check that Colab says "
                         "'Connected' (top right), then run this cell again.")
try:
    import paminaw
except ModuleNotFoundError:
    raise SystemExit("❌ paminaw is not installed. Fix: outside Colab, run "
                     "`pip install -e .` in the repo folder (see docs/OFFLINE.md).") from None

SETUP = paminaw.setup_workshop(smoke=SMOKE_TEST)

## 1 · Load & see (about 20 minutes)

A microphone measures air pressure thousands of times per second. Each measurement is a
**sample**, and the number per second is the **sample rate**. All our models expect
**16,000 samples per second (16 kHz)**. In PyTorch, a recording is a **tensor** of shape
`(channels, samples)`.

A **mel spectrogram** shows which frequencies are loud at each moment, grouped into 80
bands spaced the way human hearing spaces them. Taking the **log** makes quiet and loud
parts visible together. Whisper does not read the raw waveform; it reads exactly this
log-mel picture.

In [ ]:
import os

try:
    import paminaw
except ModuleNotFoundError:
    raise SystemExit("❌ Run cell 0 (Setup) at the top of the notebook first.") from None

# Lab 1 setup: safe to run even if you skipped the cells above
import matplotlib.pyplot as plt
import torch
import torchaudio
from IPython.display import Audio, display

from paminaw.plotting import PALETTE, plot_mel, plot_waveform, set_style

SMOKE_TEST = os.environ.get("PAMINAW_SMOKE") == "1"
paminaw.set_seed()  # 20260814, the seed from the paper
set_style()  # large fonts for the projector
N = 2 if SMOKE_TEST else 100
tl = paminaw.load_subset("tagalog", n=N)
ceb = paminaw.load_subset("cebuano", n=N)
print(f"{len(tl)} Tagalog + {len(ceb)} Cebuano clips ready.")
tl[["fleurs_id", "duration_s", "transcription"]].head(3)

### The slide code: from a file to a log-mel spectrogram

Five lines take us from a file on disk to what the model sees.

In [ ]:
path = tl.path[0]
print("Tagalog:", tl.transcription[0])

try:
    wav, sr = torchaudio.load(path)  # file → tensor + sample rate
except Exception:  # torchaudio.load needs TorchCodec + FFmpeg
    print("ℹ️ torchaudio.load is not available here, so we use paminaw.load_audio "
          "(same result).")
    wav, sr = paminaw.load_audio(path)

wav = wav.mean(dim=0, keepdim=True)  # stereo → mono, shape (1, samples)
if sr != 16_000:
    wav = torchaudio.functional.resample(wav, sr, 16_000)
mel = torchaudio.transforms.MelSpectrogram(sample_rate=16_000, n_fft=400, hop_length=160, n_mels=80)(wav)
log_mel = torch.log(mel + 1e-6)

print(f"sample rate: {sr} Hz · waveform {tuple(wav.shape)} · {wav.shape[1] / 16_000:.1f} s")
print(f"mel spectrogram {tuple(mel.shape)} = (channels, mel bands, frames)")

**Reading the shapes.** `n_fft=400` looks at 400 samples (25 ms) at a time and
`hop_length=160` moves 160 samples (10 ms) each step, so a 10-second clip becomes about
1,000 frames of 80 numbers each. `+ 1e-6` keeps `log` away from `log(0)` in silent frames.

### ✏️ Exercise 1 · Plot one Tagalog and one Cebuano waveform

Change `CLIP` to look at other recordings. 🤔 Can you see the pauses between phrases?
Which recording is longer?

In [ ]:
CLIP = 0  # 👉 try another number (0–99)

wav_tl = paminaw.load_16k_mono(tl.path[CLIP])
wav_ceb = paminaw.load_16k_mono(ceb.path[CLIP])

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
plot_waveform(wav_tl, title=f"Tagalog: {tl.transcription[CLIP][:55]}…", ax=axes[0])
plot_waveform(wav_ceb, title=f"Cebuano: {ceb.transcription[CLIP][:55]}…", ax=axes[1],
              color=PALETTE[1])
plt.tight_layout()
plt.show()

In [ ]:
# Solution: durations, and where the pauses are (frames quieter than 2% of the peak)
for name, w in [("Tagalog", wav_tl), ("Cebuano", wav_ceb)]:
    frames = w[0, : w.shape[1] // 160 * 160].reshape(-1, 160).abs().max(dim=1).values
    quiet = (frames < 0.02 * frames.max()).float().mean().item()
    print(f"{name}: {w.shape[1] / 16_000:.1f} s long, {quiet:.0%} of 10 ms frames are near-silent")

**Answer.** Pauses show up as flat stretches near zero, mostly between phrases and at the
start and end of the recording. The length depends on the sentence and on the speaker;
FLEURS sentences are read aloud, so they come out at roughly 10–25 seconds.

### 🎧 Exercise 2 · Listen at 16 kHz and at 8 kHz

Put your earphones on. Telephone audio is often 8 kHz: half as many samples, and nothing
above 4 kHz survives. 🤔 Which sounds change the most?

In [ ]:
SECONDS = 6  # play only the first few seconds
clip16 = wav_ceb[:, : 16_000 * SECONDS]
clip8 = torchaudio.functional.resample(clip16, 16_000, 8_000)
print(f"16 kHz: {tuple(clip16.shape)} samples → 8 kHz: {tuple(clip8.shape)} samples")

print("16 kHz (what the models expect):")
display(Audio(clip16.numpy(), rate=16_000))
print("8 kHz (telephone quality):")
display(Audio(clip8.numpy(), rate=8_000))

**Answer.** The 8 kHz version sounds muffled, like a phone call. Hissy consonants such as
*s*, *sh*, *f* and *t* lose most of their energy, because much of it lies above 4 kHz.

### ✏️ Exercise 3 · Compare their mel spectrograms

We resample the 8 kHz clip back to 16 kHz, so both pictures have the same axes.
🤔 What happened to the top of the picture?

In [ ]:
back16 = torchaudio.functional.resample(clip8, 8_000, 16_000)  # same length, less detail

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
plot_mel(paminaw.log_mel(paminaw.mel_spectrogram(clip16)), title="Recorded at 16 kHz", ax=axes[0])
plot_mel(paminaw.log_mel(paminaw.mel_spectrogram(back16)), title="Through 8 kHz", ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# Solution: how much energy is left above 4 kHz? (mel bands ≥ 60 of 80 ≈ above 4 kHz)
hi16 = paminaw.mel_spectrogram(clip16)[0, 60:].sum() / paminaw.mel_spectrogram(clip16)[0].sum()
hi8 = paminaw.mel_spectrogram(back16)[0, 60:].sum() / paminaw.mel_spectrogram(back16)[0].sum()
print(f"Share of energy above ~4 kHz: 16 kHz {hi16:.2%} · through 8 kHz {hi8:.4%}")

# Bonus: Tagalog vs Cebuano side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
plot_mel(paminaw.log_mel(paminaw.mel_spectrogram(wav_tl)), title="Tagalog", ax=axes[0])
plot_mel(paminaw.log_mel(paminaw.mel_spectrogram(wav_ceb)), title="Cebuano", ax=axes[1])
plt.tight_layout()
plt.show()

**Answer.** Everything above about 4 kHz (mel band ≈ 60 of 80) goes dark. A sample rate
of 8 kHz can only represent frequencies up to half that, 4 kHz (the *Nyquist* limit). A
model trained on 16 kHz audio, like Whisper, has never seen pictures with that hole, which
is one reason ASR does worse on phone recordings. Tagalog and Cebuano look alike as
pictures: same speakers' voices, same kinds of sounds. The difference is in the words,
which is why the language a model was trained on matters so much in Lab 2.